# 🦀 Day 3: Async I/O

Today: async file operations and TCP networking with Tokio.

---

## tokio::fs — Async File Operations

Tokio provides async versions of file I/O. Use these instead of `std::fs` in async code so you don't block the runtime.

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

In [ ]:
use tokio::fs;
use tokio::runtime::Runtime;

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let path = std::env::temp_dir().join("async-hello.txt");
    fs::write(&path, "Hello from async!").await.unwrap();
    
    // Read it back
    let content = fs::read_to_string(&path).await.unwrap();
    println!("{}", content);
});

## tokio::io — AsyncReadExt and AsyncWriteExt

For more control, use `AsyncReadExt` and `AsyncWriteExt` traits. They provide methods like `read_to_string`, `write_all`, etc.

In [ ]:
use tokio::io::{AsyncReadExt, AsyncWriteExt};
use tokio::fs::File;

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let path = std::env::temp_dir().join("async-bytes.txt");
    let mut file = File::create(&path).await.unwrap();
    file.write_all(b"Hello, bytes!").await.unwrap();
    file.flush().await.unwrap();
    drop(file);
    
    let mut file = File::open(&path).await.unwrap();
    let mut buf = Vec::new();
    file.read_to_end(&mut buf).await.unwrap();
    println!("{}", String::from_utf8(buf).unwrap());
});

## Buffered I/O

`tokio::io::BufReader` and `BufWriter` reduce syscalls for many small reads/writes.

In [ ]:
use tokio::io::{BufReader, BufWriter, AsyncWriteExt};
use tokio::fs::File;

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let path = std::env::temp_dir().join("async-buffered.txt");
    let file = File::create(&path).await.unwrap();
    let mut writer = BufWriter::new(file);
    writer.write_all(b"Line 1\nLine 2\nLine 3").await.unwrap();
    writer.flush().await.unwrap();
});

## tokio::net — TCP Basics

`TcpListener` accepts connections; `TcpStream` is a connected socket. Great for building servers and clients.

In [ ]:
use tokio::net::{TcpListener, TcpStream};
use tokio::io::{AsyncReadExt, AsyncWriteExt};

// Simple echo server concept: accept connection, read, write back
// In a real server you'd loop: while let Ok((stream, _)) = listener.accept().await

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let listener = TcpListener::bind("127.0.0.1:0").await.unwrap();
    let addr = listener.local_addr().unwrap();
    
    let server = tokio::spawn(async move {
        let (mut stream, _) = listener.accept().await.unwrap();
        let mut buf = [0u8; 64];
        let n = stream.read(&mut buf).await.unwrap();
        stream.write_all(&buf[..n]).await.unwrap();
    });
    
    let client = tokio::spawn(async move {
        tokio::time::sleep(std::time::Duration::from_millis(50)).await;
        let mut stream = TcpStream::connect(addr).await.unwrap();
        stream.write_all(b"Echo me!").await.unwrap();
        let mut buf = [0u8; 64];
        let n = stream.read(&mut buf).await.unwrap();
        String::from_utf8_lossy(&buf[..n]).to_string()
    });
    
    let (_, echoed) = tokio::join!(server, client);
    println!("Echoed: {}", echoed.unwrap());
});

## 🏋️ Exercise

In [ ]:
// Exercise: Async file processor.
// Create a temp file with some lines of text, then use tokio::fs::read_to_string
// to read it, count the lines, and print the count.

// YOUR CODE HERE

## 🎯 Key Takeaways

1. **tokio::fs** — `read_to_string`, `write` for simple file I/O
2. **AsyncReadExt / AsyncWriteExt** — `read_to_end`, `write_all`, etc.
3. **BufReader / BufWriter** — buffered I/O for efficiency
4. **TcpListener / TcpStream** — async TCP server and client
5. Always use async I/O in async code — blocking syscalls stall the runtime!

---
**Tomorrow:** `tokio::select!` and `tokio::join!` — racing futures and timeouts! ⏱️